In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import copy
import os
import time
import sys
import pandas as pd
from sklearn.preprocessing import StandardScaler
import seaborn as sns    
from mpl_toolkits.mplot3d import Axes3D
from sklearn.model_selection import train_test_split



In [9]:

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def lstm_cell_forward(xt, a_prev, c_prev, parameters):
    """
    Arguments:
    xt -- Données d'entrée à time-step t, array de forme (n_x, m)
    a_prev -- Etat caché précédent, array de forme (n_a, m)
    c_prev -- Etat de la cellule précédente, array de forme (n_a, m)
    parameters -- Dictionnaire Python contenant:
                Wf -- Poids de la forget gate, array de forme (n_a, n_a + n_x)
                bf -- Biais de la forget gate, array de forme (n_a, 1)
                Wi -- Poids de l'update gate, array de forme (n_a, n_a + n_x)
                bi -- Biais de l'update gate, array de forme (n_a, 1)
                Wc -- Poids de la première "tanh", array de forme (n_a, n_a + n_x)
                bc -- Biais de la première "tanh", array de forme (n_a, 1)
                Wo -- Poids de l'output gate, array de forme (n_a, n_a + n_x)
                bo -- Biais de l'output gate, array de forme (n_a, 1)
                Wy -- Poids pour l'état caché, array de forme (n_y, n_a)
                by -- Biais pour l'état caché, array de forme (n_y, 1)

    Returns:
    a_next -- Prochain état caché, array de forme (n_a, m)
    c_next -- Prochain état de cellule, array de forme (n_a, m)
    yt_pred -- Prédiction à time-step t, array de forme (n_y, m)
    cache -- Tuple de valeurs pour la backpropagation
    """

    # Récupérer les paramètres du dictionnaire
    Wf = parameters["Wf"]
    bf = parameters["bf"]
    Wi = parameters["Wi"]
    bi = parameters["bi"]
    Wc = parameters["Wc"]
    bc = parameters["bc"]
    Wo = parameters["Wo"]
    bo = parameters["bo"]
    Wy = parameters["Wy"]
    by = parameters["by"]

    # Récupérer les dimensions
    n_x, m = xt.shape
    n_y, n_a = Wy.shape

    # Concaténer a_prev et xt
    concat = np.zeros((n_a + n_x, m))
    concat[: n_a, :] = a_prev
    concat[n_a:, :] = xt

    # Calculer les valeurs pour ft, it, cct, c_next, ot, a_next
    ft = sigmoid(np.matmul(Wf, concat) + bf)
    it = sigmoid(np.matmul(Wi, concat) + bi)
    cct = np.tanh(np.matmul(Wc, concat) + bc)
    c_next = (ft * c_prev) + (it * cct)
    ot = sigmoid(np.matmul(Wo, concat) + bo)
    a_next = ot * np.tanh(c_next)

    # Calculer la prédiction
    yt_pred = softmax(np.matmul(Wy, a_next) + by)

    # Stocker les valeurs pour la backpropagation
    cache = (a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters)

    return a_next, c_next, yt_pred, cache

def lstm_cell_backward(da_next, dc_next, cache):
    """
    Arguments:
    da_next -- Gradient du prochain état caché, array de forme (n_a, m)
    dc_next -- Gradient du prochain état de cellule, array de forme (n_a, m)
    cache -- Cache du forward pass

    Returns:
    gradients -- Dictionnaire contenant les gradients
    """

    # Récupérer les informations du cache
    (a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters) = cache

    # Récupérer les dimensions
    n_x, m = xt.shape
    n_a, m = a_next.shape

    # Calculer les dérivées des portes
    dot = da_next * np.tanh(c_next) * ot * (1 - ot)
    dcct = (dc_next * it + ot * (1 - np.square(np.tanh(c_next))) * it * da_next) * (1 - np.square(cct))
    dit = (dc_next * cct + ot * (1 - np.square(np.tanh(c_next))) * cct * da_next) * it * (1 - it)
    dft = (dc_next * c_prev + ot * (1 - np.square(np.tanh(c_next))) * c_prev * da_next) * ft * (1 - ft)

    concat = np.concatenate((a_prev, xt), axis=0)

    # Calculer les dérivées des paramètres
    dWf = np.dot(dft, concat.T)
    dWi = np.dot(dit, concat.T)
    dWc = np.dot(dcct, concat.T)
    dWo = np.dot(dot, concat.T)
    dbf = np.sum(dft, axis=1, keepdims=True)
    dbi = np.sum(dit, axis=1, keepdims=True)
    dbc = np.sum(dcct, axis=1, keepdims=True)
    dbo = np.sum(dot, axis=1, keepdims=True)

    # Calculer les dérivées par rapport à l'état caché précédent, l'état de cellule précédent et l'entrée
    da_prev = np.dot(parameters['Wf'][:, :n_a].T, dft) + np.dot(parameters['Wi'][:, :n_a].T, dit) + np.dot(
        parameters['Wc'][:, :n_a].T, dcct) + np.dot(parameters['Wo'][:, :n_a].T, dot)
    dc_prev = dc_next * ft + ot * (1 - np.square(np.tanh(c_next))) * ft * da_next
    dxt = np.dot(parameters['Wf'][:, n_a:].T, dft) + np.dot(parameters['Wi'][:, n_a:].T, dit) + np.dot(
        parameters['Wc'][:, n_a:].T, dcct) + np.dot(parameters['Wo'][:, n_a:].T, dot)

    # Sauvegarder les gradients
    gradients = {"dxt": dxt, "da_prev": da_prev, "dc_prev": dc_prev, "dWf": dWf, "dbf": dbf, "dWi": dWi, "dbi": dbi,
                 "dWc": dWc, "dbc": dbc, "dWo": dWo, "dbo": dbo}

    return gradients

def lstm_forward(x, a0, parameters):
    """
    Arguments:
    x -- Données d'entrée pour chaque time-step, array de forme (n_x, m, T_x)
    a0 -- État caché initial, array de forme (n_a, m)
    parameters -- Dictionnaire Python des paramètres LSTM

    Returns:
    a -- États cachés pour chaque time-step, array de forme (n_a, m, T_x)
    y -- Prédictions pour chaque time-step, array de forme (n_y, m, T_x)
    c -- États de cellule pour chaque time-step, array de forme (n_a, m, T_x)
    caches -- Tuple de valeurs pour la backpropagation
    """

    # Initialiser les caches
    caches = []

    # Récupérer les dimensions
    n_x, m, T_x = x.shape
    n_y, n_a = parameters["Wy"].shape

    # Initialiser a, c et y avec des zéros
    a = np.zeros((n_a, m, T_x))
    c = a.copy()
    y = np.zeros((n_y, m, T_x))

    # Initialiser a_next et c_next
    a_next = a0
    c_next = np.zeros(a_next.shape)

    # Boucle sur tous les time-steps
    for t in range(T_x):
        # Mettre à jour a_next, c_next, calculer la prédiction et obtenir le cache
        a_next, c_next, yt, cache = lstm_cell_forward(x[:, :, t], a_next, c_next, parameters)
        # Sauvegarder les valeurs
        a[:, :, t] = a_next
        y[:, :, t] = yt
        c[:, :, t] = c_next
        # Ajouter le cache
        caches.append(cache)

    # Stocker les valeurs pour la backpropagation
    caches = (caches, x)

    return a, y, c, caches

def lstm_backward(da, caches):
    """
    Arguments:
    da -- Gradient par rapport aux états cachés, array de forme (n_a, m, T_x)
    caches -- Cache du forward pass

    Returns:
    gradients -- Dictionnaire Python contenant les gradients
    """

    # Récupérer les valeurs du cache
    (caches, x) = caches
    (a1, c1, a0, c0, f1, i1, cc1, o1, x1, parameters) = caches[0]

    # Récupérer les dimensions
    n_a, m, T_x = da.shape
    n_x, m = x1.shape

    # Initialiser les gradients
    dx = np.zeros((n_x, m, T_x))
    da0 = np.zeros((n_a, m))
    da_prevt = np.zeros(da0.shape)
    dc_prevt = np.zeros(da0.shape)
    dWf = np.zeros((n_a, n_a + n_x))
    dWi = np.zeros(dWf.shape)
    dWc = np.zeros(dWf.shape)
    dWo = np.zeros(dWf.shape)
    dbf = np.zeros((n_a, 1))
    dbi = np.zeros(dbf.shape)
    dbc = np.zeros(dbf.shape)
    dbo = np.zeros(dbf.shape)

    # Boucle sur la séquence en sens inverse
    for t in reversed(range(T_x)):
        # Calculer tous les gradients à l'aide de lstm_cell_backward
        gradients = lstm_cell_backward(da[:, :, t] + da_prevt, dc_prevt, caches[t])
        # Stocker ou ajouter les gradients aux gradients de l'étape précédente
        dx[:, :, t] = gradients["dxt"]
        dWf += gradients["dWf"]
        dWi += gradients["dWi"]
        dWc += gradients["dWc"]
        dWo += gradients["dWo"]
        dbf += gradients["dbf"]
        dbi += gradients["dbi"]
        dbc += gradients["dbc"]
        dbo += gradients["dbo"]
        da_prevt = gradients["da_prev"]
        dc_prevt = gradients["dc_prev"]

    # Définir le premier gradient d'activation
    da0 = gradients["da_prev"]

    # Stocker les gradients dans un dictionnaire Python
    gradients = {"dx": dx, "da0": da0, "dWf": dWf, "dbf": dbf, "dWi": dWi, "dbi": dbi,
                 "dWc": dWc, "dbc": dbc, "dWo": dWo, "dbo": dbo}

    return gradients

def initialize_adam_for_lstm(parameters):
    """
    Initialise v et s pour les paramètres du LSTM.

    Arguments:
    parameters -- Dictionnaire Python contenant les paramètres du LSTM.

    Returns:
    v -- Dictionnaire Python qui contiendra la moyenne mobile exponentielle du gradient.
    s -- Dictionnaire Python qui contiendra la moyenne mobile exponentielle du carré du gradient.
    """
    v = {}
    s = {}

    # Initialiser v, s pour tous les paramètres du LSTM
    for key in parameters.keys():
        v["d" + key] = np.zeros_like(parameters[key])
        s["d" + key] = np.zeros_like(parameters[key])

    return v, s

def update_parameters_with_adam_for_lstm(parameters, grads, v, s, t, learning_rate=0.01,
                                         beta1=0.9, beta2=0.999, epsilon=1e-8):

    v_corrected = {}  # Estimation du premier moment corrigée du biais
    s_corrected = {}  # Estimation du second moment corrigée du biais

    # Effectuer la mise à jour Adam sur tous les paramètres
    for key in parameters.keys():
        # Clé correspondante dans les dictionnaires grads, v, s
        d_key = "d" + key

        # S'assurer que nous avons le gradient correspondant
        if d_key not in grads:
            continue

        # Moyenne mobile des gradients
        v[d_key] = beta1 * v[d_key] + (1 - beta1) * grads[d_key]

        # Calcul de l'estimation du premier moment corrigée du biais
        v_corrected[d_key] = v[d_key] / (1 - beta1**t)

        # Moyenne mobile des carrés des gradients
        s[d_key] = beta2 * s[d_key] + (1 - beta2) * (grads[d_key]**2)

        # Calcul de l'estimation du second moment corrigée du biais
        s_corrected[d_key] = s[d_key] / (1 - beta2**t)

        # Mise à jour des paramètres
        parameters[key] = parameters[key] - learning_rate * v_corrected[d_key] / (np.sqrt(s_corrected[d_key]) + epsilon)

    return parameters, v, s

def initialize_lstm_parameters(n_a, n_x, n_y):
    """
    Initialise les paramètres du LSTM.

    Arguments:
    n_a -- nombre d'unités dans la couche cachée
    n_x -- taille d'entrée
    n_y -- taille de sortie

    Returns:
    parameters -- dictionnaire Python contenant les paramètres initialisés
    """
    np.random.seed(1)

    # Initialisation avec He/Xavier
    Wf = np.random.randn(n_a, n_a + n_x) * np.sqrt(1. / (n_a + n_x))
    bf = np.zeros((n_a, 1))
    Wi = np.random.randn(n_a, n_a + n_x) * np.sqrt(1. / (n_a + n_x))
    bi = np.zeros((n_a, 1))
    Wc = np.random.randn(n_a, n_a + n_x) * np.sqrt(1. / (n_a + n_x))
    bc = np.zeros((n_a, 1))
    Wo = np.random.randn(n_a, n_a + n_x) * np.sqrt(1. / (n_a + n_x))
    bo = np.zeros((n_a, 1))
    Wy = np.random.randn(n_y, n_a) * np.sqrt(1. / n_a)
    by = np.zeros((n_y, 1))

    parameters = {"Wf": Wf, "bf": bf, "Wi": Wi, "bi": bi, "Wc": Wc, "bc": bc, "Wo": Wo, "bo": bo, "Wy": Wy, "by": by}

    return parameters

def train_lstm(X_train, Y_train, n_a, n_x, n_y, num_epochs=10, seed=1, learning_rate=0.01, initial_params=None):
    """
    Entraîne un LSTM sur les données fournies, avec possibilité d'initialiser avec des paramètres existants.

    Arguments:
    X_train -- données d'entrée, numpy array de forme (n_x, m, T_x)
    Y_train -- étiquettes, numpy array de forme (n_y, m, T_x)
    n_a -- nombre d'unités dans la couche cachée
    n_x -- taille d'entrée
    n_y -- taille de sortie
    num_epochs -- nombre d'époques d'entraînement
    seed -- graine pour la reproductibilité
    learning_rate -- taux d'apprentissage
    initial_params -- paramètres initiaux (optionnel)

    Returns:
    parameters -- paramètres finaux
    parameters_history -- historique des paramètres à chaque époque
    loss_history -- historique des pertes
    """
    np.random.seed(seed)

    # Initialisation des paramètres
    if initial_params is None:
        parameters = initialize_lstm_parameters(n_a, n_x, n_y)
    else:
        parameters = copy.deepcopy(initial_params)

    parameters_history = []
    loss_history = []

    # Initialiser Adam
    v, s = initialize_adam_for_lstm(parameters)
    t = 0  # Compteur pour Adam

    for epoch in range(num_epochs):
        print(f"Époque {epoch+1}/{num_epochs}")

        # Forward pass
        a0 = np.zeros((n_a, X_train.shape[1]))
        a, y_pred, c, caches = lstm_forward(X_train, a0, parameters)

        # Calcul de la perte (cross-entropy)
        loss = -np.sum(Y_train * np.log(y_pred + 1e-8)) / (Y_train.shape[1] * Y_train.shape[2])
        loss_history.append(loss)
        print(f"Loss: {loss:.4f}")

        # Initialisation du gradient de sortie
        da = np.zeros_like(a)

        # Créer un dictionnaire complet pour les gradients
        gradients = {}

        # Pour chaque pas de temps, calculer le gradient
        dWy = np.zeros_like(parameters["Wy"])
        dby = np.zeros_like(parameters["by"])

        for t_idx in range(Y_train.shape[2]):
            # Gradient de la cross-entropy
            dy = y_pred[:, :, t_idx] - Y_train[:, :, t_idx]
            # Accumuler les gradients pour Wy et by
            dWy += np.dot(dy, a[:, :, t_idx].T)
            dby += np.sum(dy, axis=1, keepdims=True)
            # Gradient par rapport à a
            da[:, :, t_idx] = np.dot(parameters["Wy"].T, dy)

        # Backward pass pour le reste des paramètres LSTM
        lstm_gradients = lstm_backward(da, caches)

        # Combiner tous les gradients
        gradients = lstm_gradients.copy()
        gradients["dWy"] = dWy
        gradients["dby"] = dby

        # Mise à jour des paramètres avec Adam
        t += 1
        parameters, v, s = update_parameters_with_adam_for_lstm(parameters, gradients, v, s, t, learning_rate)

        # Sauvegarde des paramètres après cette époque
        parameters_history.append(copy.deepcopy(parameters))

    return parameters, parameters_history, loss_history

In [10]:

def load_jena_climate_data(file_path=None, n_clients=3, n_seeds_per_client=1, n_x=8, n_y=1, 
                          sequence_length=10, test_size=0.2, target_column='T (degC)'):
    """
    Charge et prépare le Jena Climate Dataset pour l'apprentissage fédéré LSTM.
    """
    print("Chargement du Jena Climate Dataset")
    
    # Recherche automatique du fichier
    if file_path is None:
        possible_paths = [
            "./jena_climate_2009_2016.csv",
            "./data/jena_climate_2009_2016.csv",
            "./datasets/jena_climate_2009_2016.csv",
            "jena_climate_2009_2016.csv"
        ]
        
        for path in possible_paths:
            if os.path.exists(path):
                file_path = path
                break
    
    if file_path is None or not os.path.exists(file_path):
        raise FileNotFoundError(f"Dataset non trouvé. Téléchargez jena_climate_2009_2016.csv depuis Kaggle")
    
    # Chargement et nettoyage
    df = pd.read_csv(file_path)
    print(f"Données brutes: {df.shape}")
    
    df_clean = clean_jena_data(df, target_column)
    feature_columns = select_jena_features(df_clean, n_x, target_column)
    
    clients_data = create_federated_splits(df_clean, feature_columns, target_column, 
                                         n_clients, n_seeds_per_client, sequence_length)
    
    return clients_data

def clean_jena_data(df, target_column='T (degC)'):
    """Nettoie et prépare les données."""
    df_clean = df.copy()
    
    # Conversion de date si présente
    if 'Date Time' in df_clean.columns:
        df_clean['Date Time'] = pd.to_datetime(df_clean['Date Time'], format='%d.%m.%Y %H:%M:%S')
        df_clean = df_clean.set_index('Date Time')
    
    # Interpolation des valeurs manquantes
    numeric_columns = df_clean.select_dtypes(include=[np.number]).columns
    df_clean[numeric_columns] = df_clean[numeric_columns].interpolate(method='linear')
    df_clean = df_clean.dropna()
    
    # Vérification de la colonne cible
    if target_column not in df_clean.columns:
        numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            target_column = numeric_cols[0]
            print(f"Colonne cible changée pour: {target_column}")
    
    print(f"Données nettoyées: {df_clean.shape}")
    return df_clean

def select_jena_features(df, n_x, target_column):
    """Sélectionne les meilleures features."""
    priority_features = [
        'p (mbar)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)',
        'wd (deg)', 'rh (%)', 'Tdew (degC)', 'VPmax (mbar)',
        'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)'
    ]
    
    # Features disponibles par priorité
    available_features = [f for f in priority_features if f in df.columns and f != target_column]
    
    # Ajouter d'autres colonnes numériques si nécessaire
    if len(available_features) < n_x:
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if col not in available_features and col != target_column:
                available_features.append(col)
                if len(available_features) >= n_x:
                    break
    
    selected_features = available_features[:n_x]
    
    if len(selected_features) < n_x:
        raise ValueError(f"Features insuffisantes: {len(selected_features)} < {n_x}")
    
    print(f"Features sélectionnées: {selected_features}")
    return selected_features

def create_federated_splits(df, feature_columns, target_column, n_clients, n_seeds_per_client, sequence_length):
    """Crée des divisions fédérées temporelles."""
    print(f"Création de {n_clients} clients avec {n_seeds_per_client} seeds chacun")
    
    total_samples = len(df)
    samples_per_client = total_samples // n_clients
    clients_data = []
    
    for client_id in range(n_clients):
        start_idx = client_id * samples_per_client
        end_idx = total_samples if client_id == n_clients - 1 else (client_id + 1) * samples_per_client
        
        client_df = df.iloc[start_idx:end_idx].copy()
        
        # Extraction et normalisation
        X_client = client_df[feature_columns].values
        y_client = client_df[target_column].values.reshape(-1, 1)
        
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        X_normalized = scaler_X.fit_transform(X_client)
        y_normalized = scaler_y.fit_transform(y_client)
        
        client_seeds_data = []
        
        # Création des seeds
        for seed_id in range(n_seeds_per_client):
            if n_seeds_per_client > 1:
                seed_size = len(X_normalized) // n_seeds_per_client
                seed_start = seed_id * seed_size
                seed_end = (seed_id + 1) * seed_size if seed_id < n_seeds_per_client - 1 else len(X_normalized)
                X_seed = X_normalized[seed_start:seed_end]
                y_seed = y_normalized[seed_start:seed_end]
            else:
                X_seed = X_normalized
                y_seed = y_normalized
            
            # Création des séquences
            X_sequences, y_sequences = create_sequences_jena(X_seed, y_seed, sequence_length)
            
            if len(X_sequences) > 0:
                # Format LSTM (n_x, batch_size, sequence_length)
                X_lstm = np.transpose(X_sequences, (2, 0, 1))
                y_lstm = np.transpose(y_sequences, (1, 0, 2))
                client_seeds_data.append((X_lstm, y_lstm))
        
        if client_seeds_data:
            clients_data.append(client_seeds_data)
    
    print(f"{len(clients_data)} clients créés")
    return clients_data

def create_sequences_jena(X, y, sequence_length):
    """Crée des séquences temporelles."""
    if len(X) < sequence_length:
        return np.array([]), np.array([])
    
    X_sequences = []
    y_sequences = []
    
    for i in range(len(X) - sequence_length):
        X_sequences.append(X[i:i + sequence_length])
        y_sequences.append(y[i + sequence_length])
    
    return np.array(X_sequences), np.array(y_sequences)

def create_jena_test_dataset(clients_data, batch_size, n_x, n_y, sequence_length):
    """Crée un dataset de test à partir des données clients."""
    print("Création du dataset de test")
    
    test_samples_X = []
    test_samples_Y = []
    
    for client_seeds_data in clients_data:
        if len(client_seeds_data) > 0:
            X, Y = client_seeds_data[0]
            n_samples = X.shape[1]
            n_test = min(batch_size // len(clients_data), n_samples // 3)
            
            if n_test > 0:
                test_samples_X.append(X[:, :n_test, :])
                test_samples_Y.append(Y[:, :n_test, :])
    
    if test_samples_X:
        X_test = np.concatenate(test_samples_X, axis=1)
        Y_test = np.concatenate(test_samples_Y, axis=1)
        
        # Ajustement de la taille du batch
        if X_test.shape[1] > batch_size:
            X_test = X_test[:, :batch_size, :]
            Y_test = Y_test[:, :batch_size, :]
        
        print(f"Dataset de test: {X_test.shape} -> {Y_test.shape}")
        return X_test, Y_test
    else:
        raise ValueError("Impossible de créer le dataset de test")